```{tip}
**For the presenter:** before starting the slideshow, run *Kernel → Restart Kernel and Run All Cells*, so all widgets are shown.
```

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
from IPython.display import display
# Load a fresh version of the batch-elution model from the fractionation lesson.
%run ../resources/process_optimization_setup.py

# 04.4 Process Optimization

## Contents

**Tools:** CADET-GUI process diagram and CADET-Process (Python).

**Python used:** Parameter targets, evaluator dependencies, objective directions and result selection.

**Introduces:** nested simulation/fractionation evaluation, productivity maximization, purity constraints and cyclic stationarity.

1. Choose process variables
2. Simulate and optimize collection windows
3. Evaluate one candidate
4. Optimize and verify the selected process

The characterized model parameters stay fixed. In Fractionation, collection windows changed while the process stayed fixed. Now **feed duration** and **cycle time** vary, and suitable collection windows are determined for each candidate.

The goal is higher average productivity for **A and B**, with each product pool at **at least 95% purity**.

## Choose the process variables

The setup is the two-protein Langmuir batch elution from the fractionation lesson: a **60 cm × 2.4 cm** column at **60 mL/min**. The baseline feeds for **1 min** in a **10 min** cycle.

Vary cycle time between **8 and 15 min**, and feed duration between **20 and 90 s**. The model uses seconds, so convert minute values with `* 60`.

In [ ]:
# Create an independent process for this optimization study.
process = batch_elution_process()
from CADETProcess.optimization import OptimizationProblem
optimization_problem = OptimizationProblem("batch_elution")
optimization_problem.add_evaluation_object(process)

In [ ]:
# Show the model connections and the baseline feed/elution cycle.
display(process_setup_view(process))
_ = plot_feed_cycle(process)

The dark segment is **feed duration**; the complete bar is **cycle time**. Changing these settings changes the amount injected and the time available for separation. Column and binding parameters stay fixed.

In [ ]:
# Process variables


At least **2 min** must lie between the end of feeding and the next injection. The linear constraint expresses **feed duration − cycle time ≤ −120 s**. It also remains useful if the variable bounds are widened later.

In [ ]:
# Minimum time between feeds


## Simulate repeated operation

A shorter cycle may leave material in the column for the next injection. With **cyclic stationarity** enabled, the simulator repeats cycles until successive chromatograms meet its convergence criteria.

The final cycle supplies the collection calculation. Reaching a fixed number of cycles alone would not establish stationarity.

In [ ]:
from CADETProcess.simulator import Cadet
process_simulator = Cadet()
process_simulator.evaluate_stationarity = True
optimization_problem.add_evaluator(process_simulator)

## Determine the collection windows

`FractionationOptimizer` is registered as the next evaluator, using the **95% purity** requirement introduced in Fractionation. It chooses cut times for the simulated chromatogram. As recommended there, it requests a small safety margin (**0.951**): the fractionation solver may stop just below its target, and the process constraint below must still be met.

**Cycle time and feed duration → simulation → collection windows → performance**

The process optimizer changes operating conditions; the fractionation optimizer adjusts collection windows inside each evaluation.

In [ ]:
from CADETProcess.fractionation import FractionationOptimizer
fractionation_optimizer = FractionationOptimizer()
fractionation_optimizer.optimizer.catol = 1e-4  # Absolute purity-constraint tolerance.
optimization_problem.add_evaluator(
    fractionation_optimizer,
    kwargs={
        "purity_required": [0.951, 0.951],  # Safety margin above the 95% constraint.
        "allow_empty_fractions": False,
        "ignore_failed": False,
    },
)

## Maximize productivity at the required purity

`Productivity(ranking=[1, 1])` returns the average productivity of A and B. Positive productivity should be **maximized**, so set `minimize=False`.

The explicit purity constraint uses `comparison_operator="ge"`, meaning **greater than or equal to**. The objective and constraint share the same simulation and fractionation results. The fractionation solver uses a purity-constraint tolerance of **0.0001**, so its achieved purity can fall slightly below its target; the 0.951 target keeps candidates above the 0.95 constraint. Without the margin, every candidate violates the constraint by a tiny amount and the search finds no acceptable operating point.

In [ ]:
# Productivity objective and purity constraint


## Check one candidate first

First, we evaluate the **10 min cycle / 60 s feed** baseline. The objective should be positive and both pool purities should be just above **0.95**.

This check catches a wrong parameter path or objective direction before running a full search.

In [ ]:
baseline = [10 * 60, 60]
baseline_productivity = optimization_problem.evaluate_objectives(baseline)
baseline_purities = optimization_problem.evaluate_nonlinear_constraints(baseline)
print("Baseline productivity:", baseline_productivity)
print("Baseline pool purities:", baseline_purities)

The shaded collection windows show which parts of the chromatogram enter each product pool. The remaining signal goes to waste. This baseline is kept for comparison with the selected process.

In [ ]:
# Keep the baseline chromatogram and product pools for the final comparison.
baseline_simulation = process_simulator.simulate(process)
baseline_fractionation = fractionation_optimizer.optimize_fractionation(
    baseline_simulation, purity_required=[0.951, 0.951],  # [A, B]
    allow_empty_fractions=False,
)
_ = baseline_fractionation.plot_fraction_signal()
plt.show()

## Run a short search

We use `U_NSGA3` with **8 candidates** and **3 generations**. `n_max_iter` is the common iteration-limit option; for this population optimizer, it limits generations.

This is a small demonstration run; its progress shows whether a larger evaluation budget is needed.

For `U_NSGA3`, `x0` contains the **whole starting population**. `create_initial_values()` generates eight candidates within the bounds and linear constraints; replace the first with our checked baseline.

In [ ]:
from CADETProcess.optimization import U_NSGA3
optimizer = U_NSGA3()
optimizer.n_cores = 1
optimizer.pop_size = 8
optimizer.n_max_iter = 3
optimizer.seed = 42
# Supply all eight starting candidates, including the baseline.
initial_population = optimization_problem.create_initial_values(
    optimizer.pop_size, seed=optimizer.seed,
)
initial_population[0] = baseline
results = optimizer.optimize(optimization_problem, x0=initial_population)
print("Selected [cycle time, feed duration] / s:", results.x)
print("Average productivity:", results.f)
_ = results.plot_convergence("objectives")
plt.show()

## Verify the selected process

The selected values are applied explicitly and simulated again to stationarity, showing their collection windows and performance. An operating point is accepted only when purity and recovery are acceptable alongside productivity.

In [ ]:
# Apply the selected operating point and show the resulting product pools.
process.cycle_time = float(results.x[0, 0])
process.feed_duration.time = float(results.x[0, 1])
selected_simulation = process_simulator.simulate(process)
selected_fractionation = fractionation_optimizer.optimize_fractionation(
    selected_simulation, purity_required=[0.951, 0.951],
    allow_empty_fractions=False,
)
print(selected_fractionation.performance)
_ = selected_fractionation.plot_fraction_signal()
plt.show()

## Compare with the baseline

**Feed duration and cycle time** differ first, and with them the collected products. The objective rewards higher average **productivity**; both product pools must still meet **95% purity**. **Recovery** shows how much of each fed component is collected and helps explain the productivity result.

In [ ]:
# Compare the actual baseline and selected settings, productivity, purity and recovery.
_ = plot_process_comparison(
    baseline, results.x[0], baseline_fractionation, selected_fractionation,
)

```{note}
This short search selects the best acceptable candidate it found. It does not establish a global optimum. A purity displayed as 95.0% is rounded; inspect the numerical values when checking the requirement.
```

```{dropdown} Separate productivity and recovery objectives
A trade-off study adds `Productivity()` and `Recovery()` as separate objectives, each with `n_objectives=2`, `minimize=False` and the same `requires` chain, and keeps the purity lower bound. A multi-objective result offers several trade-offs rather than a single best operating point.
```

## Summary

- Operating variables change the process; simulation and fractionation turn each candidate into performance metrics.
- Maximize productivity while keeping both product pools within the purity requirement.
- Simulate the selected settings again and compare their collection windows and performance with the baseline.

**Now:** **04.4 Process Optimization: Exercises**: adapt the problem to a single product and compare purity requirements.

**Further material:** the verified setup is a starting point for a larger search or another process. Additional optimization techniques are in **99.4 Optimization Basics** and **99.5 Advanced Optimization**; typical problems and their checks are collected in **99.6 Common Problems**.

**Next lesson:** **01.4 Contributing** shows how to ask questions, report problems and contribute to CADET.